# Camada Silver — ecommerce_pedidos

## Objetivo

Aplicar as regras de Data Quality sobre os dados provenientes da camada Bronze
da tabela `ecommerce_pedidos`.

O processamento deve permitir:

- identificar registros válidos;
- identificar registros rejeitados;
- registrar os motivos de rejeição;
- preparar os dados certificados para persistência na camada Silver.

Fluxo:

Bronze → Regras de Data Quality → Válidos / Rejeitados → Silver

In [0]:
# Leitura da camada Bronze

df_bronze = spark.table(
    "workspace.bronze.ecommerce_pedidos"
)

print("Registros recebidos da Bronze:", df_bronze.count())

df_bronze.printSchema()

In [0]:
# Define os valores permitidos para campos com domínio fechado.

status_validos = [
    "Processando",
    "Pagamento Aprovado",
    "Em Separação",
    "Enviado",
    "Entregue",
    "Cancelado"
]

metodos_pagamento_validos = [
    "cartão crédito",
    "cartão débito",
    "pix",
    "boleto"
]

In [0]:
# Aplica as primeiras regras de Data Quality e cria indicadores de falha por registro.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

janela_id_pedido = Window.partitionBy("id_pedido")

df_dq = (
    df_bronze

    # Quantidade de ocorrências de cada id_pedido
    .withColumn(
        "qtd_id_pedido",
        F.count("*").over(janela_id_pedido)
    )

    #id_pedido não pode ser nulo nem duplicado;
    .withColumn(
        "dq_id_pedido_nulo",
        F.col("id_pedido").isNull()
    )

    .withColumn(
        "dq_id_pedido_duplicado",
        F.col("id_pedido").isNotNull()
        & (F.col("qtd_id_pedido") > 1)
    )

    # 2. id_cliente não pode ser nulo;
    .withColumn(
        "dq_id_cliente_nulo",
        F.col("id_cliente").isNull()
    )

    # 3. status_pedido deve estar no domínio permitido;
    .withColumn(
        "dq_status_invalido",
        F.col("status_pedido").isNull()
        | ~F.col("status_pedido").isin(status_validos)
    )

    # 4. valor_total deve ser maior que zero;
    .withColumn(
        "dq_valor_total_invalido",
        F.col("valor_total").isNull()
        | (F.col("valor_total") <= 0)
    )

    # 5. metodo_pagamento deve estar no domínio permitido.
    .withColumn(
        "dq_metodo_pagamento_invalido",
        F.col("metodo_pagamento").isNull()
        | ~F.col("metodo_pagamento").isin(
            metodos_pagamento_validos
        )
    )
)

In [0]:
# Exibe uma amostra dos registros já avaliados pelas regras de Data Quality.

display(
    df_dq.limit(20)
)